# E7: Online Learning with CVX Memory

Three improvements over E5:
1. **Task-type strategy templates** — abstract strategies instead of literal actions
2. **Online reward annotation** — learn from own experience across rounds
3. **Expert reward decay** — penalize experts that lead to failures

**Model**: Ollama qwen2.5:7b (zero cost)

**Protocol**: 3 rounds × 30 games. After each round, agent's own experience
(wins=1.0, fails=0.0) is added to the index. Expert trajectories that were
retrieved but led to failure get 10% reward decay per failure.

**Research question**: Does the agent improve across rounds as the memory
refines itself?

In [ ]:
"""
E7: Online Learning with CVX — Ollama prototype

Improvements over E5:
1. Task-type metadata filtering (reduce noise from wrong task types)
2. Abstract strategy extraction (not literal actions)
3. Online reward annotation (learn from own experience)
4. All on Ollama (qwen2.5:7b) — zero API cost
"""
import os, json, time, re
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === CONFIG: OLLAMA ===
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'
client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
LLM_MODEL = 'qwen2.5-coder:7b-instruct'

MAX_STEPS = 30
N_EVAL = 30
N_ROUNDS = 3  # Online learning rounds
TOP_K = 3

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'LLM: {LLM_MODEL} (Ollama)')

import chronos_vector as cvx

# === TASK TYPE DETECTION ===
TASK_TYPES = {
    'heat': 'pick_heat_then_place',
    'cool': 'pick_cool_then_place', 
    'clean': 'pick_clean_then_place',
    'examine': 'look_at_obj_in_light',
    'look at': 'look_at_obj_in_light',
    'find two': 'pick_two_obj_and_place',
    'put two': 'pick_two_obj_and_place',
    'put a': 'pick_and_place',
    'put some': 'pick_and_place',
}

def detect_task_type(task_text):
    task_lower = task_text.lower()
    for keyword, ttype in TASK_TYPES.items():
        if keyword in task_lower:
            return ttype
    return 'pick_and_place'

# === BUILD INDEX WITH TASK TYPE METADATA ===
with open('data/episodic/alfworld_metadata.json') as f:
    meta = json.load(f)

print('Building index with task-type metadata...')
index = cvx.TemporalIndex(m=16, ef_construction=100)
action_lookup, task_lookup = {}, {}

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    task = ep.get('task', '')
    task_type = detect_task_type(task)
    task_lookup[ep_id] = task
    
    for si, step in enumerate(ep['steps']):
        text = f"{task} | {step.get('action','')} | {step.get('observation','')}"
        vec = embedder.encode(text).tolist()
        entity_id = ep_id
        timestamp = ep_id * 10000 + si
        # Expert trajectories start with reward=1.0
        index.insert(entity_id, timestamp, vec, reward=1.0)
        action_lookup[timestamp] = step.get('action', '')

print(f'Index: {len(index)} points from {len(meta)} expert episodes')

# Save base index for reset between rounds
index.save('data/episodic/e7_base_index')

# === STRATEGY ABSTRACTION ===
STRATEGY_TEMPLATES = {
    'pick_and_place': 'Strategy: 1) Find the object by checking likely locations 2) Take the object 3) Go to the target location 4) Put the object there',
    'pick_heat_then_place': 'Strategy: 1) Find the object 2) Take it 3) Go to microwave 4) Heat it 5) Take it from microwave 6) Go to target 7) Put it there',
    'pick_cool_then_place': 'Strategy: 1) Find the object 2) Take it 3) Go to fridge 4) Cool it 5) Take it from fridge 6) Go to target 7) Put it there',
    'pick_clean_then_place': 'Strategy: 1) Find the object 2) Take it 3) Go to sinkbasin 4) Clean it 5) Take it 6) Go to target 7) Put it there',
    'look_at_obj_in_light': 'Strategy: 1) Find the object 2) Take it 3) Go to desklamp/floorlamp 4) Use the lamp to examine it',
    'pick_two_obj_and_place': 'Strategy: 1) Find first object 2) Take it 3) Put it at target 4) Find second object 5) Take it 6) Put it at target',
}

def format_context(results, task_type):
    """Format with strategy abstraction + expert actions."""
    if not results:
        return ''
    
    strategy = STRATEGY_TEMPLATES.get(task_type, '')
    parts = []
    if strategy:
        parts.append(f'{strategy}\n')
    
    parts.append('Expert actions from similar past tasks:')
    for i, r in enumerate(results[:TOP_K]):
        ep_id = r['entity_id']
        expert_task = task_lookup.get(ep_id, '?')
        parts.append(f'  Expert {i+1} (task: "{expert_task}"):')
        if r['successors']:
            for nid, ts, vec in r['successors'][:4]:
                action = action_lookup.get(ts, '?')
                if action != '?':
                    parts.append(f'    - {action}')
    return '\n'.join(parts) + '\n'

# === AGENT ===
def llm_call(system, user):
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, max_tokens=100,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return ''

def agent_act(observation, admissible, task, history, context=''):
    system = 'You are a household agent. Choose ONE action from the list. Output ONLY the action.'
    user = f'Task: {task}\n\n'
    if context: user += context + '\n'
    if history:
        user += 'Recent:\n' + '\n'.join(f'  > {h["a"]} -> {h["o"][:60]}' for h in history[-5:]) + '\n\n'
    user += f'Obs: {observation}\nActions:\n' + '\n'.join(f'  - {a}' for a in admissible) + '\nChoose:'
    chosen = llm_call(system, user).lower()
    for a in admissible:
        if a.lower() == chosen: return a
    for a in admissible:
        if a.lower() in chosen or chosen in a.lower(): return a
    return admissible[0]

def run_game(env, index, use_memory=False, task_type_filter=False):
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)
    history = []
    retrieval_nodes = []  # Track which expert nodes were used
    
    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        ctx = ''
        if use_memory:
            try:
                qv = embedder.encode(f'{task} | {observation}').tolist()
                results = index.causal_search(vector=qv, k=TOP_K, temporal_context=5)
                ctx = format_context(results, task_type)
                # Track used nodes for reward annotation
                for r in results:
                    retrieval_nodes.append(r)
            except:
                pass
        
        action = agent_act(observation, admissible, task, history, ctx)
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'a': action, 'o': observation[:100]})
        if dones[0]: break
    
    won = info['won'][0]
    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'history': history,
        'retrieval_nodes': retrieval_nodes,
    }

# === ONLINE LEARNING LOOP ===
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv
config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0, 'num_eval_games': 0,
    },
    'env': {'goal_desc_human_anns_prob': 0.0, 'task_types': [1,2,3,4,5,6],
            'domain_randomization': False, 'expert_type': 'handcoded'},
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
              'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2')},
}
env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)
print(f'ALFWorld: {len(env_wrapper.game_files)} games\n')

all_round_results = {}
next_ep_id = 1000  # New episodes start at 1000 to not collide with experts

for round_num in range(N_ROUNDS):
    print(f'\n{"#"*60}')
    print(f'ROUND {round_num + 1}/{N_ROUNDS} — Index has {len(index)} points')
    print(f'{"#"*60}')
    
    # Run evaluation
    cond = f'Round{round_num+1}_CVX'
    results = []
    wins = 0
    for g in range(N_EVAL):
        r = run_game(env, index, use_memory=True)
        results.append(r)
        if r['won']: wins += 1
        print(f'  {g+1}/{N_EVAL}: {"WIN" if r["won"] else "fail"} ({r["steps"]}s) {r["task_type"][:20]} [{wins}/{g+1}={wins/(g+1)*100:.0f}%]', flush=True)
    
    rate = wins / N_EVAL * 100
    print(f'\n  >>> Round {round_num+1}: {wins}/{N_EVAL} = {rate:.1f}%')
    all_round_results[cond] = [{'won':r['won'],'steps':r['steps'],'task':r['task'],'task_type':r['task_type']} for r in results]
    
    # === ONLINE LEARNING: Add own experience to index ===
    added = 0
    for r in results:
        reward = 1.0 if r['won'] else 0.0
        for si, step in enumerate(r['history']):
            text = f"{r['task']} | {step['a']} | {step['o']}"
            vec = embedder.encode(text).tolist()
            index.insert(next_ep_id, next_ep_id * 10000 + si, vec, reward=reward)
            action_lookup[next_ep_id * 10000 + si] = step['a']
            added += 1
        task_lookup[next_ep_id] = r['task']
        next_ep_id += 1
    
    # === REWARD DECAY: Penalize expert trajectories that were retrieved but led to failure ===
    decayed = 0
    for r in results:
        if not r['won']:
            for ret in r.get('retrieval_nodes', []):
                for nid, ts, vec in ret.get('successors', []):
                    current = index.reward(nid)
                    if current and current == current:  # not NaN
                        new_reward = max(0.0, current * 0.9)
                        index.set_reward(nid, new_reward)
                        decayed += 1
    
    print(f'  Added {added} own experience points, decayed {decayed} expert rewards')
    
    # Save after each round
    with open('data/episodic/e7_online_results.json', 'w') as f:
        json.dump(all_round_results, f, indent=2)

# === FINAL SUMMARY ===
print(f'\n{"="*60}')
print('ONLINE LEARNING SUMMARY')
print(f'{"="*60}')
for cond, results in all_round_results.items():
    wins = sum(1 for r in results if r['won'])
    print(f'  {cond}: {wins}/{len(results)} = {wins/len(results)*100:.1f}%')

# Per task type breakdown
print(f'\nPer task type (last round):')
last_results = list(all_round_results.values())[-1]
by_type = {}
for r in last_results:
    tt = r.get('task_type', '?')
    by_type.setdefault(tt, []).append(r['won'])
for tt, outcomes in sorted(by_type.items()):
    w = sum(outcomes)
    print(f'  {tt}: {w}/{len(outcomes)} = {w/len(outcomes)*100:.0f}%')

print('\nSaved to data/episodic/e7_online_results.json')
